## Terra Bank Loan Risk Analysis 
#### Phase 1: Data Cleaning and Preprocessing

#### About Dataset:

***Source*** Kaggle - [Bank Loan Data] [https://www.kaggle.com/datasets/udaymalviya/bank-loan-data]

*** Data Description *** :

- 45,000 rows each representing a unique personal bank loan
- 14 columns covering client demographics, financial profile, loan characteristics, and credit history. 
- each record corresponnds to a closed loan with an outcome of either fully prepaid or defaulted 

#### Project Goal:

- To Identify and better understand which applicant and loan features are most strongly associated with a loan being repaid versus defaulted
- Engineer a predictive model that cam help with reducing default risk, adjusting approval thresholds, etc.



### Objectives
- Import necessary libraries and packages
- Load and examine the NFL play-by-play dataset
- Identify and retain relevant features for 4th down analysis
- Initial dataset cleaning and preparation for Preliminary Analysis 
?

In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
import joblib
import xgboost as xgb
import os
from pathlib import Path

# current working directory is notebooks/cleaning_eda
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent  # go up two levels: cleaning_eda -> notebooks -> project root

DATA_FILE = PROJECT_ROOT / "data" / "raw" / "bank_loan_data.csv"

df = pd.read_csv(DATA_FILE)

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 14 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_age                      45000 non-null  float64
 1   person_gender                   45000 non-null  object 
 2   person_education                45000 non-null  object 
 3   person_income                   45000 non-null  float64
 4   person_emp_exp                  45000 non-null  int64  
 5   person_home_ownership           45000 non-null  object 
 6   loan_amnt                       45000 non-null  float64
 7   loan_intent                     45000 non-null  object 
 8   loan_int_rate                   45000 non-null  float64
 9   loan_percent_income             45000 non-null  float64
 10  cb_person_cred_hist_length      45000 non-null  float64
 11  credit_score                    45000 non-null  int64  
 12  previous_loan_defaults_on_file  

In [3]:
# clean column names: strip whitespace, convert to lowercase, replace spaces with underscores

df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

In [4]:
# check for duplicates

duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

if duplicate_count > 0:
    display(df[df.duplicated(keep=False)].sort_index())

Duplicate rows: 0


In [5]:
# check for missing values

df.isnull().sum()

person_age                        0
person_gender                     0
person_education                  0
person_income                     0
person_emp_exp                    0
person_home_ownership             0
loan_amnt                         0
loan_intent                       0
loan_int_rate                     0
loan_percent_income               0
cb_person_cred_hist_length        0
credit_score                      0
previous_loan_defaults_on_file    0
loan_status                       0
dtype: int64

In [6]:
new_column_names = {
    "person_age": "age",
    'person_gender': "gender",
    "person_income": "income",
    "person_education": "education",
    "person_emp_exp": "employment_experience_length",
    "person_home_ownership": "home_ownership",
    "loan_amnt": "loan_amount",
    "loan_intent": "loan_purpose",
    "loan_int_rate": "loan_interest_rate",
    "loan_percent_income": "loan_percent_to_income",
    "cb_person_cred_hist_length": "credit_history_length",
    "credit_score": "credit_score",
    "previous_loan_defaults_on_file": "previous_loan_default",
    "loan_status": "loan_repaid" 
}

df = df.rename(columns=new_column_names)

In [7]:
df.head()


,age,gender,education,income,employment_experience_length,home_ownership,loan_amount,loan_purpose,loan_interest_rate,loan_percent_to_income,credit_history_length,credit_score,previous_loan_default,loan_repaid
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [8]:
# check for potential duplicate loans by creating a composite key by combining multiple columns that together should uniquely identify a loan application

key_cols = [
    "age",
    "gender",
    "education",
    "income",
    "employment_experience_length",
    "home_ownership",
    "loan_amount",
    "loan_purpose",
    "loan_interest_rate"
]

df["loan_composite_key"] = df[key_cols].astype(str).agg("|".join, axis=1)

df["is_dup_key"] = df.duplicated(subset="loan_composite_key", keep=False)

dupes_key = df[df["is_dup_key"]].sort_values("loan_composite_key")
dupes_key.head(20)

,age,gender,education,income,employment_experience_length,home_ownership,loan_amount,loan_purpose,loan_interest_rate,loan_percent_to_income,credit_history_length,credit_score,previous_loan_default,loan_repaid,loan_composite_key,is_dup_key


In [9]:
## Check for outliers in numeric columns 

# Define Numeric columns to check for outliers

num_cols = [
    "age", 
    "income",
    'loan_amount',
    'loan_interest_rate',
    'loan_percent_to_income',
    'credit_history_length',
    'credit_score',
] 

df[num_cols].describe().T.round(0)


,count,mean,std,min,25%,50%,75%,max
age,45000.0,28.0,6.0,20.0,24.0,26.0,30.0,144.0
income,45000.0,80319.0,80422.0,8000.0,47204.0,67048.0,95789.0,7200766.0
loan_amount,45000.0,9583.0,6315.0,500.0,5000.0,8000.0,12237.0,35000.0
loan_interest_rate,45000.0,11.0,3.0,5.0,9.0,11.0,13.0,20.0
loan_percent_to_income,45000.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
credit_history_length,45000.0,6.0,4.0,2.0,3.0,4.0,8.0,30.0
credit_score,45000.0,633.0,50.0,390.0,601.0,640.0,670.0,850.0


In [10]:

# Establish rule thresholds for key numerical features 
MAX_AGE = 90
MIN_AGE = 18
MAX_INCOME = 1_000_000  # adjust to your market
MAX_EXPERIENCE_BUFFER = 14  # assume work starts no earlier than age 14

# 2a. Create rule-based flags
df["age_outlier_rule"] = (df["age"] < MIN_AGE) | (df["age"] > MAX_AGE)

df["income_outlier_rule"] = df["income"] > MAX_INCOME

df["experience_outlier_rule"] = (
    df["employment_experience_length"] > (df["age"] - MAX_EXPERIENCE_BUFFER)
)

df["credit_history_outlier_rule"] = (
    df["credit_history_length"] > (df["age"] - 16)
)

# 2b. Quick summary of how many rows are flagged by each rule
rule_flags = [
    "age_outlier_rule",
    "income_outlier_rule",
    "experience_outlier_rule",
    "credit_history_outlier_rule",
]

# 2c. Proportion of rows flagged by each rule
rule_flag_summary = (
    df[rule_flags]
    .mean()
    .sort_values(ascending=False)
    .to_frame(name="flagged_proportion")
)

display(rule_flag_summary)

,flagged_proportion
income_outlier_rule,0.000533
age_outlier_rule,0.000178
experience_outlier_rule,0.000000
credit_history_outlier_rule,0.000000


In [11]:
rule_flag_counts = df[rule_flags].sum().to_frame(name="flagged_count")
display(rule_flag_counts)

,flagged_count
age_outlier_rule,8
income_outlier_rule,24
experience_outlier_rule,0
credit_history_outlier_rule,0


In [12]:

outlier_iqr_props = {}

for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    flag_col = f"{col}_outlier_iqr"
    df[flag_col] = (df[col] < lower) | (df[col] > upper)

    outlier_iqr_props[col] = df[flag_col].mean()

    # Absolute counts for each feature's IQR outliers
outlier_iqr_counts = {
    col: df[f"{col}_outlier_iqr"].sum()
    for col in outlier_iqr_props.keys()

}

In [13]:
iqr_props_summary = (
    pd.Series(outlier_iqr_props)
    .to_frame(name="iqr_outlier_proportion")
)

iqr_both_summary = (
    iqr_props_summary
    .join(
        pd.Series(outlier_iqr_counts)
        .to_frame(name="iqr_outlier_count")
    )
    .sort_values("iqr_outlier_count", ascending=False)
)

display(iqr_both_summary)

,iqr_outlier_proportion,iqr_outlier_count
loan_amount,0.052178,2348
income,0.049289,2218
age,0.048622,2188
credit_history_length,0.030356,1366
loan_percent_to_income,0.016533,744
credit_score,0.010378,467
loan_interest_rate,0.002756,124


In [14]:
# Optional: Z‑score flags for a few features
# For roughly symmetric variables like credit_score, loan_interest_rate, and age, add Z‑score flags.

from scipy import stats

z_thresh = 3.0
z_cols = ["credit_score", "loan_interest_rate", "age"]

for col in z_cols:
    zscores = stats.zscore(df[col].astype(float), nan_policy="omit")
    flag_col = f"{col}_outlier_z"
    df[flag_col] = np.abs(zscores) > z_thresh

In [15]:
summary_rows = []

for col in num_cols:
    row = {"feature": col}

    # Rule-based, if defined
    rule_col = f"{col}_outlier_rule"
    if rule_col in df.columns:
        row["rule_outlier_pct"] = round(df[rule_col].mean() * 100, 2)
    else:
        row["rule_outlier_pct"] = np.nan

    # IQR-based
    iqr_col = f"{col}_outlier_iqr"
    if iqr_col in df.columns:
        row["iqr_outlier_pct"] = round(df[iqr_col].mean() * 100, 2)
    else:
        row["iqr_outlier_pct"] = np.nan

    # Z-score-based
    z_col = f"{col}_outlier_z"
    if z_col in df.columns:
        row["z_outlier_pct"] = round(df[z_col].mean() * 100, 2)
    else:
        row["z_outlier_pct"] = np.nan

    summary_rows.append(row)

outlier_summary = pd.DataFrame(summary_rows)


### Cleaning Findings

- all columns have been standardized 
- no duplicates detected
- no missing values detected 
- cleaned column names for clarity 
- target feature is "loan_repaid" with 1 being yes and 0 being the loan defaulted

### Basic Data statistics 

- Average Age is 27 Years
- Average Income is 80,319
- Average loan amount is 9583
- Average interest rate is 11%
- Average history length is 6 years 
- Average credit score is 633

### Established the following rules for numerical thresholds for spotting loan based situations:
- MAX_AGE = 90
- MIN_AGE = 18
- MAX_INCOME = 1_000_000  # adjust to your market
- MAX_EXPERIENCE_BUFFER = 14  # assume work starts no earlier than age 14

### See Table below to review outlier percentages for each numeroca; feature by method 

In [16]:
# Outlier Findings Percentage Summary for Rule, IQR, and Z‑score Methods

display(outlier_summary)

,feature,rule_outlier_pct,iqr_outlier_pct,z_outlier_pct
0,age,0.02,4.86,1.69
1,income,0.05,4.93,NaN
2,loan_amount,NaN,5.22,NaN
3,loan_interest_rate,NaN,0.28,0.19
4,loan_percent_to_income,NaN,1.65,NaN
5,credit_history_length,NaN,3.04,NaN
6,credit_score,NaN,1.04,0.52


In [17]:
df.head(10)

,age,gender,education,income,employment_experience_length,home_ownership,loan_amount,loan_purpose,loan_interest_rate,loan_percent_to_income,...,age_outlier_iqr,income_outlier_iqr,loan_amount_outlier_iqr,loan_interest_rate_outlier_iqr,loan_percent_to_income_outlier_iqr,credit_history_length_outlier_iqr,credit_score_outlier_iqr,credit_score_outlier_z,loan_interest_rate_outlier_z,age_outlier_z
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,...,False,False,True,False,True,False,False,False,False,False
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,...,False,False,False,False,False,False,False,False,False,False
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,...,False,False,False,False,True,False,False,False,False,False
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,...,False,False,True,False,True,False,False,False,False,False
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,...,False,False,True,False,True,False,False,False,False,False
5,21.0,female,High School,12951.0,0,OWN,2500.0,VENTURE,7.14,0.19,...,False,False,False,False,False,False,False,False,False,False
6,26.0,female,Bachelor,93471.0,1,RENT,35000.0,EDUCATION,12.42,0.37,...,False,False,True,False,False,False,False,False,False,False
7,24.0,female,High School,95550.0,5,RENT,35000.0,MEDICAL,11.11,0.37,...,False,False,True,False,False,False,False,False,False,False
8,24.0,female,Associate,100684.0,3,RENT,35000.0,PERSONAL,8.90,0.35,...,False,False,True,False,False,False,False,False,False,False
9,21.0,female,High School,12739.0,0,OWN,1600.0,VENTURE,14.74,0.13,...,False,False,False,False,False,False,False,False,False,False


In [31]:
# Go up two levels from notebooks/cleaning_eda to project root, then into data/interim
interim_dir = Path("../../data/interim")
interim_dir.mkdir(parents=True, exist_ok=True)  # creates bank-loan-risk/data/interim if needed

output_path = interim_dir / "bank_loan_data_v2.csv"
df.to_csv(output_path, index=False)

print(f"Saved to: {output_path.resolve()}")

Saved to: /Users/macbook/projects/data-science/bank-loan-risk/data/interim/bank_loan_data_v2.csv
